In [1]:
# AD based differential expression first
import pandas as pd
import os

# Base DE files directory
base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/Differential_Expression_Final/Validation/"
files = [
    "poisson_DE_results_In.csv",
    "poisson_DE_results_Mic.csv",
    "poisson_DE_results_Oli.csv",
    "poisson_DE_results_Ast.csv",
    "poisson_DE_results_Ex.csv"
]

# Map short codes to pretty names
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Ex': 'Excitatory Neurons'
}

# Output workbook path
out_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/Validation/Validation_Differential_Expression.xlsx"

# Create an Excel writer
with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    # Add title page
    title_text = (
        "This workbook contains the full differential expression analysis results for our validation cohort between Alzheimer's samples and controls."
        "Poisson model for Alzheimer's disease vs. Control across major brain cell types. "
        "Sheets correspond to individual cell types."
    )
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="Title_Page", index=False)
    
    # Add each cell type sheet
    for file in files:
        cell_short = file.replace("poisson_DE_results_", "").replace(".csv", "")
        cell_name = ct_labels[cell_short]
        
        # Load data
        df = pd.read_csv(os.path.join(base_dir, file))

        df["DEG"] = (df["p_adj"] < 0.05)
        
        # Write to sheet
        df.to_excel(writer, sheet_name=cell_name, index=False)

print(f"Workbook saved to: {out_path}")

Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/Validation/Validation_Differential_Expression.xlsx


In [2]:
import os
import joblib
from collections import defaultdict
import pandas as pd
from scipy.stats import fisher_exact

# === Load background genes ===
gene_matrix = pd.read_parquet('/home/adm808/NormalizedCellMatrixSyn18485175.parquet')
background_genes = set(gene_matrix.index.str.upper())
print(f"Total background genes: {len(background_genes)}")

# === Define cell types ===
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Ex': 'Excitatory Neurons'
}
cell_types = list(ct_labels.keys())

# === Function to collect predictive genes and counts ===
def collect_predictive_genes_with_counts(base_dir, cell_types, n_splits):
    gene_counts = {}
    for cell_type in cell_types:
        gene_presence = defaultdict(int)
        for split in range(1, n_splits + 1):
            joblib_path = os.path.join(base_dir, cell_type, f"split_{split}", "maximal_classifier.joblib")
            if not os.path.exists(joblib_path):
                continue
            model = joblib.load(joblib_path)
            feature_names = model.feature_names_in_
            importances = model.feature_importances_
            for gene, imp in zip(feature_names, importances):
                if imp > 0:
                    gene_presence[gene.upper()] += 1
        gene_counts[cell_type] = dict(gene_presence)
        print(f"{cell_type}: {len([g for g, c in gene_presence.items() if c >= 2])} predictive genes (≥2 splits)")
    return gene_counts

# === Collect predictors ===
ad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
ad_counts = collect_predictive_genes_with_counts(ad_base_dir, cell_types, 5)

cerad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Validation_Multirun_cell_on_cell_genes_Lau_rfe"
cerad_counts = collect_predictive_genes_with_counts(cerad_base_dir, cell_types, 5)

# === Fisher exact test summary table ===
results = []

# === Excel writer ===
excel_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/Validation/Validation_overlap_results.xlsx"
writer = pd.ExcelWriter(excel_path, engine='xlsxwriter')

# === Add description sheet ===
description_text = (
    "This Excel workbook summarizes the overlap of predictive genes between the original dataset "
    "and the validation dataset, analyzed across different cell types.\n\n"
    "For each cell type:\n"
    "- We collect genes that are predictive (appear in ≥2 splits).\n"
    "- We compute overlaps between the original and validation sets.\n"
    "- We perform Fisher's exact test to assess enrichment of overlap.\n\n"
    "The 'Fisher_Results' sheet summarizes overall statistics.\n"
    "Each cell-type-specific sheet lists genes, split counts, and whether the gene is concordant."
)

# Create a DataFrame with a single column to hold the text
desc_df = pd.DataFrame({'Description': description_text.split('\n')})
desc_df.to_excel(writer, sheet_name='Description', index=False)

for cell_type in cell_types:
    ad_dict = ad_counts[cell_type]
    cerad_dict = cerad_counts[cell_type]

    ad_genes = set([gene for gene, count in ad_dict.items() if count >= 2])
    cerad_genes = set([gene for gene, count in cerad_dict.items() if count >= 2])
    overlap = ad_genes & cerad_genes

    # Fisher test
    a = len(overlap)
    b = len(ad_genes - overlap)
    c = len(cerad_genes - overlap)
    d = len(background_genes - (ad_genes | cerad_genes))
    table = [[a, b], [c, d]]
    odds_ratio, p_value = fisher_exact(table, alternative='greater')

    results.append({
        'CellType': ct_labels[cell_type],
        'Original Dataset AD Predictive Genes': len(ad_genes),
        'Validation Dataset AD Predictive Genes': len(cerad_genes),
        'Overlap': a,
        'OddsRatio': odds_ratio,
        'PValue': p_value
    })

    # === Build detailed gene table for this cell type ===
    gene_rows = []
    for gene in sorted(background_genes):
        orig_splits = ad_dict.get(gene, 0)
        val_splits = cerad_dict.get(gene, 0)
        pred_orig = orig_splits >= 2
        pred_val = val_splits >= 2
        overlap_flag = pred_orig and pred_val
        gene_rows.append({
            'Gene': gene,
            'Predictive_in_Original': pred_orig,
            'Predictive_in_Validation': pred_val,
            'Overlapping': overlap_flag,
            'Original_Splits': orig_splits,
            'Validation_Splits': val_splits
        })

    gene_df = pd.DataFrame(gene_rows)
    gene_df.to_excel(writer, sheet_name=ct_labels[cell_type][:31], index=False)

# === Save Fisher summary table as first sheet ===
results_df = pd.DataFrame(results)
results_df.to_excel(writer, sheet_name='Fisher_Results', index=False)

# === Save workbook ===
writer.close()
print(f"Excel workbook saved to {excel_path}")

Total background genes: 17926
Ast: 175 predictive genes (≥2 splits)
Mic: 466 predictive genes (≥2 splits)
In: 108 predictive genes (≥2 splits)
Oli: 621 predictive genes (≥2 splits)
Ex: 162 predictive genes (≥2 splits)
Ast: 212 predictive genes (≥2 splits)
Mic: 274 predictive genes (≥2 splits)
In: 50 predictive genes (≥2 splits)
Oli: 268 predictive genes (≥2 splits)
Ex: 17 predictive genes (≥2 splits)
Excel workbook saved to /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/Validation/Validation_overlap_results.xlsx


In [3]:
import os
import joblib
import pandas as pd
import numpy as np
from collections import defaultdict
from scipy.stats import fisher_exact

# === Background genes ===
gene_matrix = pd.read_parquet('/home/adm808/NormalizedCellMatrixSyn18485175.parquet')
background_genes = set(gene_matrix.index.str.upper())
print(f"Total background genes: {len(background_genes)}")

# === Cell type labels ===
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Ex': 'Excitatory Neurons'
}
cell_types = list(ct_labels.keys())

# === Paths ===
ad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
cerad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Validation_Multirun_cell_on_cell_genes_Lau_rfe"
de_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/Differential_Expression_Final/Fixed/"
cerad_de_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/Differential_Expression_Final/Validation/"

de_files = {
    "Ast": "poisson_DE_results_Ast.csv",
    "Mic": "poisson_DE_results_Mic.csv",
    "In": "poisson_DE_results_In.csv",
    "Oli": "poisson_DE_results_Oli.csv",
    "Ex": "poisson_DE_results_Ex.csv"
}

# === Prepare Excel writer ===
output_excel_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/Validation/Validation_Fisher_Tests.xlsx"
with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:

    # --- Add title page first ---
    title_text = (
        "This workbook summarizes all Fisher tests for the Validation Cohort run per cell type.\n\n"
        "Included tests:\n"
        "1. Original (AD) predictors vs Validation predictors\n"
        "2. Validation predictors vs Validation DE genes\n"
        "3. Original (AD) DE genes vs Validation DE genes"
    )
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="Title_Page", index=False)

    # === Loop through each cell type ===
    for cell_short in cell_types:
        cell_name = ct_labels[cell_short]
        rows = []

        ### === Get AD predictors ===
        gene_presence_ad = defaultdict(int)
        for split in range(1, 6):
            path = os.path.join(ad_base_dir, cell_short, f"split_{split}", "maximal_classifier.joblib")
            if not os.path.exists(path):
                continue
            model = joblib.load(path)
            for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
                if imp > 0:
                    gene_presence_ad[gene.upper()] += 1
        ad_predictors = {gene for gene, count in gene_presence_ad.items() if count >= 2}

        ### === Get Validation predictors ===
        gene_presence_cerad = defaultdict(int)
        for split in range(1, 6):
            path = os.path.join(cerad_base_dir, cell_short, f"split_{split}", "maximal_classifier.joblib")
            if not os.path.exists(path):
                continue
            model = joblib.load(path)
            for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
                if imp > 0:
                    gene_presence_cerad[gene.upper()] += 1
        cerad_predictors = {gene for gene, count in gene_presence_cerad.items() if count >= 2}

        ### === Get AD DE genes ===
        ad_de_df = pd.read_csv(os.path.join(de_base_dir, de_files[cell_short]))
        ad_de_df["gene_upper"] = ad_de_df["gene"].str.upper()
        ad_de_genes = set(ad_de_df[ad_de_df["p_adj"] < 0.05]["gene_upper"])

        ### === Get Validation DE genes ===
        cerad_de_df = pd.read_csv(os.path.join(cerad_de_base_dir, de_files[cell_short]))
        cerad_de_df["gene_upper"] = cerad_de_df["gene"].str.upper()
        cerad_de_genes = set(cerad_de_df[cerad_de_df["p_adj"] < 0.05]["gene_upper"])

        # === Test 1: AD predictors vs Validation predictors ===
        overlap = ad_predictors & cerad_predictors
        a = len(overlap)
        b = len(ad_predictors - overlap)
        c = len(cerad_predictors - overlap)
        d = len(background_genes - (ad_predictors | cerad_predictors))
        table = [[a, b], [c, d]]
        or1, pv1 = fisher_exact(table, alternative='greater')
        rows.append({
            "Test": "Original (AD) predictors vs Validation predictors",
            "Num_Set1_Genes": len(ad_predictors),
            "Num_Set2_Genes": len(cerad_predictors),
            "Overlap": a,
            "OddsRatio": or1,
            "PValue": pv1,
            "Overlap_Genes": ", ".join(sorted(overlap))
        })

        # === Test 2: Validation predictors vs Validation DE genes ===
        overlap = cerad_predictors & cerad_de_genes
        a = len(overlap)
        b = len(cerad_predictors - overlap)
        c = len(cerad_de_genes - overlap)
        d = len(background_genes - (cerad_predictors | cerad_de_genes))
        table = [[a, b], [c, d]]
        or2, pv2 = fisher_exact(table, alternative='greater')
        rows.append({
            "Test": "Validation predictors vs Validation DE genes",
            "Num_Set1_Genes": len(cerad_predictors),
            "Num_Set2_Genes": len(cerad_de_genes),
            "Overlap": a,
            "OddsRatio": or2,
            "PValue": pv2,
            "Overlap_Genes": ", ".join(sorted(overlap))
        })

        # === Test 3: Original (AD) DE genes vs Validation DE genes ===
        overlap = ad_de_genes & cerad_de_genes
        a = len(overlap)
        b = len(ad_de_genes - overlap)
        c = len(cerad_de_genes - overlap)
        d = len(background_genes - (ad_de_genes | cerad_de_genes))
        table = [[a, b], [c, d]]
        or3, pv3 = fisher_exact(table, alternative='greater')
        rows.append({
            "Test": "Original (AD) DE genes vs Validation DE genes",
            "Num_Set1_Genes": len(ad_de_genes),
            "Num_Set2_Genes": len(cerad_de_genes),
            "Overlap": a,
            "OddsRatio": or3,
            "PValue": pv3,
            "Overlap_Genes": ", ".join(sorted(overlap))
        })

        # === Save sheet for this cell type ===
        cell_df = pd.DataFrame(rows)
        cell_df.to_excel(writer, sheet_name=cell_name[:31], index=False)

print(f"Workbook saved to: {output_excel_path}")

Total background genes: 17926
Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/Validation/Validation_Fisher_Tests.xlsx
